# Catalog the ocean-icechunks stores as STAC items

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ocean-icechunks/2026-ioos-hackathon/blob/main/notebooks/stac-geoparquet-create.ipynb)

The [ocean-icechunks](https://source.coop/ocean-icechunks) account on Source Cooperative holds
many Icechunk stores. Here we find them, read each store's spatial and temporal extent and
its variables, and write **one [STAC](https://stacspec.org) item per dataset** to a single
[stac-geoparquet](https://github.com/stac-utils/stac-geoparquet) file.

The next notebook, `stac-geoparquet-query`, searches that file by place and time. Nothing is
stored in the repository: you build the catalog yourself, from whatever is in the bucket today.
Run both notebooks in the same session, because the file is written to the local disk.

The build opens every store over the network, so it takes a minute or so.

In [ ]:
%pip install -q "icechunk>=2.2" "xarray>=2026.7" "zarr>=3.4" rustac numpy

## 1. Find the stores

The bucket is public, so we list it anonymously with plain S3 requests. An Icechunk repository
is a folder that holds a `repo` object and a `snapshots/` folder, so we walk the listing until we
reach one. The `test-repo/` folder holds scratch copies, so we skip it.

In [ ]:
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET

BUCKET, REGION = "us-west-2.opendata.source.coop", "us-west-2"
ACCOUNT = "ocean-icechunks"
NS = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}


def list_prefix(prefix):
    """Return (sub-prefixes, keys) directly under prefix."""
    query = urllib.parse.urlencode({"list-type": "2", "prefix": prefix, "delimiter": "/"})
    with urllib.request.urlopen(f"https://s3.{REGION}.amazonaws.com/{BUCKET}/?{query}", timeout=30) as r:
        root = ET.fromstring(r.read())
    return ([e.text for e in root.findall("s3:CommonPrefixes/s3:Prefix", NS)],
            [e.text for e in root.findall("s3:Contents/s3:Key", NS)])


def find_stores(prefix, depth=0, max_depth=4):
    prefixes, keys = list_prefix(prefix)
    if f"{prefix}repo" in keys and f"{prefix}snapshots/" in prefixes:
        yield prefix.removeprefix(f"{ACCOUNT}/").rstrip("/")
        return
    if depth < max_depth:
        for p in prefixes:
            if p != f"{ACCOUNT}/test-repo/":
                yield from find_stores(p, depth + 1, max_depth)


stores = list(find_stores(f"{ACCOUNT}/"))
print(len(stores), "Icechunk stores")
for s in stores:
    print(" ", s)

## 2. Read a store's metadata

We open each store anonymously and read only the small coordinate arrays, never the data
variables. A store either holds one dataset at its root, or several datasets in sub-groups
(NOAA's ocean heat content stores keep `daily`, `14day` and `14day_v1` side by side). We make one
item for each dataset.

In [ ]:
import warnings
import icechunk
import xarray as xr
import zarr

warnings.filterwarnings("ignore")


def open_store(path):
    storage = icechunk.s3_storage(bucket=BUCKET, prefix=f"{ACCOUNT}/{path}", region=REGION, anonymous=True)
    return icechunk.Repository.open(storage).readonly_session("main").store


def groups_of(store):
    """Group paths that hold data: [''] for a dataset at the root, else its sub-groups."""
    root = zarr.open_group(store, mode="r")
    if any(a.ndim >= 2 for _, a in root.arrays()):
        return [""]
    return sorted(name for name, _ in root.groups())

## 3. Work out the bounding box

STAC wants longitude in -180..180, but some grids use 0..360 (the wave model here runs 199..206
degrees east). A grid that spans the antimeridian, like the North Pacific ocean heat content
(100..280 degrees east), becomes a `bbox` with west greater than east and a `MultiPolygon`
geometry split at 180, as the STAC spec says.

In [ ]:
def wrap_lon(lon):
    return (lon + 180) % 360 - 180


def bbox_and_geometry(lon, lat):
    s, n = float(lat.min()), float(lat.max())
    w, e = float(lon.min()), float(lon.max())
    if e - w >= 359:  # global grid
        w, e = -180.0, 180.0
    else:
        w, e = wrap_lon(w), wrap_lon(e)
    if w <= e:
        return [w, s, e, n], {"type": "Polygon", "coordinates": [[[w, s], [e, s], [e, n], [w, n], [w, s]]]}
    west = [[[w, s], [180, s], [180, n], [w, n], [w, s]]]
    east = [[[-180, s], [e, s], [e, n], [-180, n], [-180, s]]]
    return [w, s, e, n], {"type": "MultiPolygon", "coordinates": [west, east]}

## 4. Build the STAC item

The item gets the bounding box, the time range (`start_datetime` and `end_datetime`), a
datacube description of the variables, and one asset that points at the Icechunk store. The
`icechunk:group` and `icechunk:branch` fields on the asset are our own convention, so a reader
knows which group to open. A climatology has no time axis, so it gets a nominal datetime only.

In [ ]:
import numpy as np
from datetime import datetime, timezone

HTTPS = "https://data.source.coop"
LON_NAMES, LAT_NAMES = ("lon", "longitude"), ("lat", "latitude")


def iso(t):
    return np.datetime_as_string(t, unit="s") + "Z"


def make_item(path, group):
    ds = xr.open_zarr(open_store(path), group=group or None, consolidated=False, chunks=None)
    lon = next(ds[c] for c in LON_NAMES if c in ds.coords)
    lat = next(ds[c] for c in LAT_NAMES if c in ds.coords)
    bbox, geometry = bbox_and_geometry(lon.values, lat.values)

    props = {"title": ds.attrs.get("title") or f"{path}{'/' + group if group else ''}",
             "description": ds.attrs.get("summary")}
    if "time" in ds.coords and ds["time"].dtype.kind == "M":
        props["start_datetime"] = iso(ds["time"].min().values)
        props["end_datetime"] = iso(ds["time"].max().values)
        props["datetime"] = props["start_datetime"]
        props["icechunk:time_steps"] = int(ds.sizes["time"])
    else:
        props["datetime"] = datetime(1970, 1, 1, tzinfo=timezone.utc).isoformat()
    props["cube:dimensions"] = {d: {"size": int(n)} for d, n in ds.sizes.items()}
    props["cube:variables"] = {
        name: {"type": "data", "dimensions": list(v.dims), "unit": v.attrs.get("units"),
               "description": v.attrs.get("long_name") or v.attrs.get("standard_name")}
        for name, v in ds.data_vars.items() if v.ndim >= 2}

    prefix = f"{ACCOUNT}/{path}"
    return {
        "type": "Feature", "stac_version": "1.1.0",
        "id": "-".join(p for p in (*path.split("/"), group) if p and p != "icechunk"),
        "collection": ACCOUNT, "geometry": geometry, "bbox": bbox,
        "properties": {k: v for k, v in props.items() if v is not None},
        "links": [],
        "assets": {"icechunk": {
            "href": f"s3://{BUCKET}/{prefix}", "title": "Icechunk repository",
            "type": "application/vnd.icechunk", "roles": ["data"],
            "icechunk:group": group or "/", "icechunk:branch": "main",
            "alternate": {"https": {"href": f"{HTTPS}/{prefix}"}}}},
    }

## 5. Build all items and write the geoparquet file

In [ ]:
items = []
for path in stores:
    for group in groups_of(open_store(path)):
        item = make_item(path, group)
        p = item["properties"]
        print(f"{item['id']:36s} {p.get('start_datetime', 'no time axis'):22s} {p.get('end_datetime', '')}")
        items.append(item)

import rustac

PARQUET = "ocean-icechunks-items.parquet"
await rustac.write(PARQUET, items)
print(f"wrote {len(items)} items to {PARQUET}")

That is the whole catalog: one file, one row per dataset. Next, open `stac-geoparquet-query`
to search it by place and time.